This notebook provides a step-by-step guide to reproducing the results of Cyclic Layout Permutation based Zero-Noise Extrapolation (CLP-ZNE) error mitigation for one of the experimental setups considered in the paper https://arxiv.org/abs/2511.02901. 

Specifically, we evaluate Sherrington-Kirkpatrick hamiltonians on a 12-qubit, 3-layer TwoLocal circuits featuring cyclic connectivity, executed on the FakeTorino backend with $T_{1}$ and $T_{2}$ relaxation times reduced by a factor of 10.

In [ ]:
from src.clp_zne.qpu_info.layout_cycles import heron_layout_cycles_12q
from src.clp_zne.hamiltonians import sherrington_kirkpatrick_model
from src.clp_zne.qpu_info.backends import FakeTorino
from src.clp_zne.mitigate import clp_zne_mitigate_1d_topology_circuit
from src.clp_zne.utils import compute_evals_ideal
from qiskit.circuit.library import TwoLocal
from tqdm import tqdm
import numpy as np
import os

In [ ]:
OUTPUT_FOLDER = r"data\FakeTorino--Cyclic circuit--12 qubits--T1 T2 noise x10--CLP ZNE 3 params 4 cycles"
N_QUBITS = 12 # This parameter is locked for the current configuration. Do not modify
N_CIRCUITS = 20
N_LAYERS = 3
N_OBSERVABLES = 100
T1_T2_NOISE_MULTIPLIER = 10 # Set to 1 to recover the original (unmodified) noise

In [ ]:
backend = FakeTorino()

observables = [sherrington_kirkpatrick_model(N_QUBITS, h=1, seed=i) for i in range(N_OBSERVABLES)]

# Initialize circuits
circuits = []
for i in range(N_CIRCUITS):
    circ = TwoLocal(N_QUBITS, ['rx', 'rz'], 'cz', entanglement="circular", reps=N_LAYERS)
    rng = np.random.default_rng(i)
    parameters = rng.uniform(-np.pi, np.pi, circ.num_parameters)
    circ.assign_parameters(parameters, inplace=True)
    circuits.append(circ)

In [ ]:
# Initial circuit layouts in different qubit cycles (four cycles in this case)
layouts = [heron_layout_cycles_12q[i] for i in [4, 5, 9, 13]]

all_evals_ideal = []
all_evals_mitigated = []
all_evals_noisy = []
all_error_sums = []

for circ in tqdm(circuits):
    # Perform error mitigation
    evals_mitigated, evals_noisy, error_sums = clp_zne_mitigate_1d_topology_circuit(circ, observables, layouts,
                                                                    backend, num_params=3,
                                                                    therm_noise_multiplier=T1_T2_NOISE_MULTIPLIER)
    
    # Compute noiseless expectation value
    evals_ideal = compute_evals_ideal(circ, observables)
    
    all_evals_ideal.append(evals_ideal)
    all_evals_mitigated.append(evals_mitigated)
    all_evals_noisy.append(evals_noisy)
    all_error_sums.append(error_sums)

In [ ]:
# Save the results
data_to_save = {
    'evals_ideal.npy': np.array(all_evals_ideal),
    'evals_mitigated.npy': np.array(all_evals_mitigated),
    'evals_noisy.npy': np.array(all_evals_noisy),
    'error_sums.npy': np.array(all_error_sums)
}

for filename, data in data_to_save.items():
    path = os.path.join(OUTPUT_FOLDER, filename)
    np.save(path, data)

print(f"Successfully saved {len(data_to_save)} files to: {OUTPUT_FOLDER}")